# Manuscript Fig. 4 — policy outcomes and dynamics

Existing formal plotting cells are collected here in manuscript panel order.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

start_dir = Path.cwd().resolve()
workbench_dir = next(
    (candidate
     for root in (start_dir, *start_dir.parents)
     for candidate in (root, root / "figure_workbench", root / "DiFCoN" / "figure_workbench")
     if (candidate / "plotting" / "__init__.py").is_file()),
    None,
)
if workbench_dir is None:
    raise FileNotFoundError("Cannot locate figure_workbench/plotting from the current directory")
if str(workbench_dir) not in sys.path:
    sys.path.insert(0, str(workbench_dir))

from plotting import subfigure_plotting_utils as sp
from plotting._paths import FIGURE_DIR
from plotting._source_tables import read_source_table
from plotting.manuscript_export import DEFAULT_FONT_SIZES as FONT_SIZES, save_main_panel
from plotting.baseline_visualization import B41BB_ACTION_CMAP as b41bb_cmap, PD1_ACTION_CMAP as pd1_cmap
from plotting.baseline_visualization import SCORE_LABELS

FONT_SIZE = FONT_SIZES["base"]
print(f"Output root: {FIGURE_DIR}")

## Cell organization

Panels follow the manuscript order; existing composite groups remain combined.
Run the setup cell before rendering panels. The shared export uses 8 pt text by default.
Exports use `Fig{number}{panel(s)}_{description}` in `figure/`: PNG (600 dpi), PDF, and SVG with text converted to paths.


## Fig. 4a-d — test-cohort outcomes

In [ ]:
outcome_font_sizes = {**FONT_SIZES, "xtick": 6.5}
fig, axes = plt.subplots(1, 4, figsize=(200 / 25.4, 65 / 25.4))
summary = sp.load_baseline_table("test_strategy_summary")
paired = read_source_table("baseline")
metrics = (
    ("final_C_rel", r"Final tumor, $C(T)/C_0$"),
    ("tumor_auc_rel", r"CTB, $\frac{1}{T-t_0}\int_{t_0}^{T} c(t)\,\mathrm{d}t$"),
    ("mean_activity", r"Mean CD8$^{+}$ T cell activity, $\overline{a}$"),
    ("total_dose", r"Total dose, $\sum(u_P+u_B)$"),
)
for ax, (metric, ylabel) in zip(axes, metrics):
    sp.cohort_metric_panel(summary, paired, metric, ax=ax, ylabel=ylabel, font_size=FONT_SIZE)

fig.subplots_adjust(left=0.055, right=0.995, top=0.97, bottom=0.20, wspace=0.42)

paths = save_main_panel(fig, "Fig4a-d_test_cohort_outcomes", font_sizes=outcome_font_sizes)
fig

## Fig. 4e — patient-by-day action heatmaps

In [ ]:
FIGSIZE = (125 / 25.4, 60 / 25.4)

# ---------------------------
# Layout settings
# ---------------------------
CELL_WIDTH = 0.95     # Width of each day cell. Smaller values increase horizontal spacing.
CELL_HEIGHT = 1    # Height of each patient cell. Smaller values increase vertical spacing.
LINEWIDTH = 0.05      # Width of the white cell borders

# ---------------------------
# Load Optimal actions for the test set.
# ---------------------------
actions = sp.load_baseline_table("test_actions")
act = actions[actions["strategy"].eq("Optimal")].copy()

# Use the same patient order in both heatmaps, sorted by descending total dose.
patient_order = (
    act.groupby("patient_id")[["action_PD1", "action_B41BB"]]
    .sum().sum(axis=1).sort_values(ascending=False).index.tolist()
)

pd1_mat = act.pivot(index="patient_id", columns="day", values="action_PD1").reindex(patient_order)
b41bb_mat = act.pivot(index="patient_id", columns="day", values="action_B41BB").reindex(patient_order)

days = pd1_mat.columns.to_numpy(float)
n_patients = len(patient_order)

# ---------------------------
# Draw one column at a time to control cell width.
# ---------------------------
def draw_action_heatmap(ax, matrix, cmap, title):
    y = np.arange(n_patients)
    y0 = y + (1 - CELL_HEIGHT) / 2
    y1 = y0 + CELL_HEIGHT

    for j, day in enumerate(days):
        vals = matrix.iloc[:, j].to_numpy(float)
        x0, x1 = day - CELL_WIDTH / 2, day + CELL_WIDTH / 2
        for i, v in enumerate(vals):
            ax.pcolormesh(
                [x0, x1], [y0[i], y1[i]], np.array([[v]]),
                cmap=cmap, vmin=0, vmax=1, shading="flat",
                edgecolors="white", linewidth=LINEWIDTH
            )

    ax.set_xlim(days.min() - 0.5, days.max() + 0.5)
    ax.set_ylim(n_patients, 0)
    ax.set_xticks(np.arange(1, 18, 2))
    ax.set_title(title, fontsize=FONT_SIZES["title"], pad=3)

    # Hide y-axis labels.
    ax.set_yticks([])
    ax.set_ylabel("", fontsize=FONT_SIZES["axis_label"])
    ax.tick_params(axis="y", left=False, labelleft=False)

    ax.tick_params(axis="x", labelsize=FONT_SIZES["xtick"], bottom=True, labelbottom=True, length=2.5, width=0.7, direction="out")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(False)

# ---------------------------
# Place the heatmaps side by side.
# ---------------------------
with sp.figure_style(FONT_SIZE):
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, sharey=True)

    draw_action_heatmap(axes[0], pd1_mat, pd1_cmap, "")
    draw_action_heatmap(axes[1], b41bb_mat, b41bb_cmap, "")

    axes[0].set_xlabel("Day", fontsize=FONT_SIZES["axis_label"])
    axes[0].set_ylabel("test patients", fontsize=FONT_SIZES["axis_label"])
    axes[1].set_xlabel("Day", fontsize=FONT_SIZES["axis_label"])

    fig.subplots_adjust(left=0.06, right=0.985, top=0.88, bottom=0.20, wspace=0.10)


paths = save_main_panel(fig, "Fig4e_patient_action_heatmaps")
fig
# print(paths)

## Fig. 4f — patient action-allocation scatter

In [ ]:
FIGSIZE = (80 / 25.4, 55 / 25.4)

fig = sp.baseline_patient_action_allocation_panel(figsize=FIGSIZE, font_size=FONT_SIZE)

paths = save_main_panel(fig, "Fig4f_patient_action_allocation")
print(paths)
fig

## Fig. 4g-j — representative-patient dynamics

Panels g-j are generated together by the single existing plotting cell below and are intentionally not split.

In [ ]:
# Select the patient with the largest reduction in objective relative to Combination.
losses = sp.load_baseline_table("test_strategy_summary").pivot(
    index="patient_id", columns="strategy", values="loss"
)
PATIENT_ID = (losses["Combine"] - losses["Optimal"]).idxmax()
print(f"Representative patient: {PATIENT_ID}")
FIGSIZE = (200 / 25.4, 125 / 25.4)

# ============================================================
# Load policy actions for the representative patient.
# ============================================================

actions = sp.load_baseline_table("test_actions")
act = actions[(actions["patient_id"].eq(PATIENT_ID)) & (actions["strategy"].eq("Optimal"))].sort_values("day").copy()
if act.empty:
    raise ValueError(f"No Optimal action data for {PATIENT_ID}")
trajectory = sp.load_baseline_table("test_trajectories")
trajectory = trajectory.loc[trajectory["patient_id"].eq(PATIENT_ID)]
if trajectory.empty:
    raise ValueError(f"No trajectory data for {PATIENT_ID}")

days = act["day"].to_numpy(float)
pd1 = act["action_PD1"].to_numpy(float)[None, :]
b41bb = act["action_B41BB"].to_numpy(float)[None, :]
x_edges = np.r_[days - 0.5, days[-1] + 0.5]

with sp.figure_style(FONT_SIZE):

    # ========================================================
    # Figure layout
    # ========================================================

    fig = plt.figure(figsize=FIGSIZE)
    gs = fig.add_gridspec(5, 2, height_ratios=[0.08, 0.09, 1.0, 0.16, 1.0], hspace=0.03, wspace=0.34)

    ax_hp, ax_hb = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    ax_c, ax_act = fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])
    ax_score_leg = fig.add_subplot(gs[3, 1]); ax_score_leg.axis("off")
    ax_n, ax_score = fig.add_subplot(gs[4, 0]), fig.add_subplot(gs[4, 1])

    # ========================================================
    # Top row shows the representative patient's action heatmaps.
    # ========================================================

    m1 = ax_hp.pcolormesh(x_edges, [0, 1], pd1, cmap=pd1_cmap, vmin=0, vmax=1, shading="flat",edgecolors="white", linewidth=0.6)
    m2 = ax_hb.pcolormesh(x_edges, [0, 1], b41bb, cmap=b41bb_cmap, vmin=0, vmax=1, shading="flat",edgecolors="white", linewidth=0.6)
    # Hide both axes to show the heatmaps as dose timelines.
    for ax, title in [(ax_hp, ""), (ax_hb, "")]:
        ax.set(xlim=(0.5, 17.5), ylim=(0, 1), xticks=[], yticks=[])
        ax.set_title(title, fontsize=FONT_SIZES["title"], fontweight="bold", pad=2)
        ax.set_xlabel("", fontsize=FONT_SIZES["axis_label"]); ax.tick_params(left=False, bottom=False)
        for side in ("top", "right", "left", "bottom"): ax.spines[side].set_visible(False)
        ax.grid(False)

    # ========================================================
    # C(t)/C(0)、activity、N(t)
    # ========================================================

    for panel, ax in (("tumor", ax_c), ("activity", ax_act), ("N", ax_n)):
        sp.representative_timeseries_panel(panel, trajectory, ax=ax, font_size=FONT_SIZE)

    # Show the Day label and tick labels only on the bottom row.
    for ax in (ax_c, ax_act):
        ax.set_xlabel("", fontsize=FONT_SIZES["axis_label"])
        ax.tick_params(axis="x", bottom=True, labelbottom=False)

    # ========================================================
    # Compare program scores under Combination and DiFCoN for the same patient.
    # ========================================================

    sp.patient_scores_panel(trajectory, ax=ax_score, font_size=FONT_SIZE)

    # Label both bottom panels Day and mark days 1, 3, ..., 17.
    for ax in (ax_n, ax_score):
        ax.set_xticks(np.arange(1, 18, 2))
        ax.set_xlabel("Day", fontsize=FONT_SIZES["axis_label"])
        ax.tick_params(axis="x", bottom=True, labelbottom=True)

    # ========================================================
    # Legend for the four treatment strategies
    # ========================================================

    handles, labels = ax_c.get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="center", bbox_to_anchor=(0.24, 0.56), ncol=2,
        frameon=False, fontsize=FONT_SIZES["legend"], handlelength=1.8,
        columnspacing=1.2, handletextpad=0.45
    , title_fontsize=FONT_SIZES["legend_title"])

    # ========================================================
    # Program score legend
    # First row shows the five programs.
    # Second row shows Combination and DiFCoN.
    # ========================================================

    score_order = ("precursor", "activation", "eff_early", "eff_cytox", "dysfunction")
    score_handles = [
        Line2D([0], [0], color=sp.PROGRAM_COLORS[s], lw=1.8,
               label=SCORE_LABELS[s])
        for s in score_order
    ]

    strategy_handles = [
        Line2D([0], [0], color="#555555", lw=1.4, ls="--", marker="o", markersize=3.8,
               markerfacecolor="#555555", markeredgecolor="#555555", label="Combination"),
        Line2D([0], [0], color="#555555", lw=1.4, ls="-", marker="s", markersize=3.8,
               markerfacecolor="#555555", markeredgecolor="#555555", label="DiFCoN"),
    ]

    leg1 = ax_score_leg.legend(
        handles=score_handles, loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=5,
        frameon=False, fontsize=FONT_SIZES["legend"], handlelength=1.3,
        columnspacing=0.55, handletextpad=0.3
    , title_fontsize=FONT_SIZES["legend_title"])
    ax_score_leg.add_artist(leg1)

    ax_score_leg.legend(
        handles=strategy_handles, loc="lower center", bbox_to_anchor=(0.5, -0.38), ncol=2,
        frameon=False, fontsize=FONT_SIZES["legend"], handlelength=4.2,
        columnspacing=1.0, handletextpad=0.4
    , title_fontsize=FONT_SIZES["legend_title"])

    # ========================================================
    # Figure margins
    # ========================================================

    fig.subplots_adjust(left=0.085, right=0.985, top=0.97, bottom=0.075)

    # ========================================================
    # Adjust the heatmap position and height.
    # ========================================================

    HEAT_Y_SHIFT = -0.03      # Positive shifts move up, and negative shifts move down.
    HEAT_HEIGHT_SCALE = 0.85   # Fraction of the original heatmap height. Smaller values make it thinner.

    for ax in (ax_hp, ax_hb):
        p = ax.get_position()
        new_h = p.height * HEAT_HEIGHT_SCALE
        new_y = p.y0 + (p.height - new_h) / 2 + HEAT_Y_SHIFT
        ax.set_position([p.x0, new_y, p.width, new_h])

    # ========================================================
    # Stack two horizontal colorbars in the gap between the heatmaps.
    # Set their dimensions and position using the gap.
    # ========================================================

    p1, p2 = ax_hp.get_position(), ax_hb.get_position()
    gap_left, gap_right = p1.x1, p2.x0
    gap_width = gap_right - gap_left

    CB_WIDTH_SCALE = 0.65    # Colorbar length as a fraction of the gap
    CB_HEIGHT = 0.02       # Colorbar height
    CB_GAP = 0.025          # Vertical gap between the colorbars
    CB_Y_SHIFT = 0.0         # Vertical shift of both colorbars. Positive values move them up.

    cb_width = gap_width * CB_WIDTH_SCALE
    cb_left = gap_left + (gap_width - cb_width -0.05) / 2
    center_y = p1.y0 + p1.height / 2 + CB_Y_SHIFT

    # Create the upper and lower horizontal colorbars.
    cax1 = fig.add_axes([cb_left, center_y + CB_GAP / 2, cb_width, CB_HEIGHT])
    cax2 = fig.add_axes([cb_left, center_y - CB_GAP / 2 - CB_HEIGHT, cb_width, CB_HEIGHT])

    cb1 = fig.colorbar(m1, cax=cax1, orientation="horizontal")
    cb2 = fig.colorbar(m2, cax=cax2, orientation="horizontal")

    cb1.ax.tick_params(bottom=False, labelbottom=False)

    # Position the colorbar labels in figure coordinates.
    # The two labels share the same horizontal position.
    LABEL_TEXT_1 = r"$u_B(t)$"
    LABEL_TEXT_2 = r"$u_P(t)$"
    LABEL_X = cb_left + cb_width / 2   # Horizontal position
    LABEL_Y_1 = center_y - 0.045         # Vertical position of the first label
    LABEL_Y_2 = center_y     # Vertical position of the second label

    fig.text(
        LABEL_X, LABEL_Y_1, LABEL_TEXT_1,
        ha="center", va="center",
        fontsize=FONT_SIZES["annotation"]
    )
    fig.text(
            LABEL_X, LABEL_Y_2, LABEL_TEXT_2,
            ha="center", va="center",
            fontsize=FONT_SIZES["annotation"]
        )
    for cb in (cb1, cb2):
        cb.set_ticks([0, 1])
        cb.ax.tick_params(labelsize=FONT_SIZES["colorbar_tick"], length=1.8, pad=1)
        cb.outline.set_linewidth(0.5)
fig
paths = save_main_panel(fig, "Fig4g-j_representative_patient_dynamics")